In [1]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

# Folder Setup 
video_path  = "input2.mp4"
folders     = ["extracted_frames", "grayscale_frames", "denoised_checked",
               "shadow_removed", "enhanced_images", "final_output",
               "marked_potholes", "crack_detected", "crack_lines"]
for f in folders:
    os.makedirs(f, exist_ok=True)

# 1: FRAME EXTRACTION

def extract_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()
    frame = cv2.resize(frame, (640, 480))
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rows  = np.where(gray.mean(axis=1) > 10)[0]
    cols  = np.where(gray.mean(axis=0) > 10)[0]
    t, b  = rows[0], rows[-1]
    l, r  = cols[0], cols[-1]

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    fc, sc = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (640, 480))
        frame = frame[t:b, l:r]
        frame = cv2.resize(frame, (640, 480))
        if fc % int(fps) == 0:
            cv2.imwrite(f"extracted_frames/frame_{sc:04d}.png", frame)
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            cv2.imwrite(f"grayscale_frames/frame_{sc:04d}.png", gray)
            sc += 1
        fc += 1
    cap.release()
    print(f"Stage 1 Done: {sc} frames extracted")


# 2: NOISE REMOVAL

def remove_noise():
    for f in sorted(os.listdir("grayscale_frames")):
        img = cv2.imread(f"grayscale_frames/{f}", 0)
        if img is None: continue
        denoised = cv2.GaussianBlur(img, (5, 5), 1)
        cv2.imwrite(f"denoised_checked/{f}", denoised)
    print("Stage 2 Done: Noise removal complete")


# 3: SHADOW REMOVAL

def remove_shadow():
    for f in sorted(os.listdir("denoised_checked")):
        img = cv2.imread(f"denoised_checked/{f}", 0)
        if img is None: continue
        illum  = cv2.GaussianBlur(img, (51, 51), 0)
        sub    = cv2.subtract(img, illum)
        result = cv2.normalize(sub, None, 0, 255, cv2.NORM_MINMAX)
        result = cv2.addWeighted(result, 0.3, img, 0.7, 0)
        cv2.imwrite(f"shadow_removed/{f}", result)
    print("Stage 3 Done: Shadow removal complete")

# 4: CONTRAST ENHANCEMENT

def enhance_contrast():
    for f in sorted(os.listdir("shadow_removed")):
        img = cv2.imread(f"shadow_removed/{f}", 0)
        if img is None: continue
        clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(img)
        enhanced = cv2.convertScaleAbs(enhanced, alpha=1.1, beta=5)
        cv2.imwrite(f"enhanced_images/{f}", enhanced)
    print("Stage 4 Done: Contrast enhancement complete")


# 5: MOTION BLUR SHARPENING

def sharpen_frames():
    for f in sorted(os.listdir("enhanced_images")):
        img = cv2.imread(f"enhanced_images/{f}", 0)
        if img is None: continue
        if cv2.Laplacian(img, cv2.CV_64F).var() < 120:
            blurred = cv2.GaussianBlur(img, (0, 0), 2)
            img     = cv2.addWeighted(img, 1.5, blurred, -0.5, 0)
        cv2.imwrite(f"final_output/{f}", img)
    print("Stage 5 Done: Motion blur sharpening complete")


# 6: POTHOLE DETECTION

def detect_potholes():
    total = 0
    for f in sorted(os.listdir("final_output")):
        img = cv2.imread(f"final_output/{f}", 0)
        if img is None: continue
        mask = (img < np.percentile(img, 8)).astype(np.uint8) * 255
        k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        marked  = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        h, w    = img.shape
        for c in cnts:
            x, y, cw, ch = cv2.boundingRect(c)
            ratio = cw / ch if ch != 0 else 0
            if cv2.contourArea(c) > 8000 and 0.5 < ratio < 2.0 and x > 50 and x+cw < w-50:
                cv2.drawContours(marked, [c], -1, (0, 255, 0), 2)
                total += 1
        cv2.imwrite(f"marked_potholes/{f}", marked)
    print(f"Stage 6 Done: Potholes detected = {total}")


# 7: EDGE CRACK DETECTION

def detect_edge_cracks():
    total  = 0
    frames = [f"frame_{i:04d}.png" for i in range(0, 22)]
    for f in frames:
        img = cv2.imread(f"final_output/{f}", 0)
        if img is None: continue
        h, w        = img.shape
        edge_region = np.zeros_like(img)
        edge_region[:, :int(w*0.15)] = img[:, :int(w*0.15)]
        edge_region[:, int(w*0.85):] = img[:, int(w*0.85):]
        mask = (edge_region < np.percentile(img, 10)) & (edge_region > 0)
        mask = mask.astype(np.uint8) * 255
        k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        marked  = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        for c in cnts:
            if cv2.contourArea(c) > 2000:
                cv2.drawContours(marked, [c], -1, (0, 0, 255), 2)
                total += 1
        cv2.imwrite(f"crack_detected/{f}", marked)
    print(f"Stage 7 Done: Edge cracks detected = {total}")


# 8: NORMAL CRACK DETECTION

def detect_normal_cracks():
    total  = 0
    frames = [f"frame_{i:04d}.png" for i in range(22, 43)]
    for f in frames:
        img  = cv2.imread(f"extracted_frames/{f}")
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        roi  = gray[int(h*0.35):, :]
        blur = cv2.GaussianBlur(roi, (3, 3), 0)
        mask = (blur < np.percentile(blur, 20)).astype(np.uint8) * 255
        k    = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        mask = cv2.dilate(mask, k, iterations=2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        marked  = img.copy()
        for c in cnts:
            area          = cv2.contourArea(c)
            x, y, cw, ch = cv2.boundingRect(c)
            ratio         = max(cw, ch) / min(cw, ch) if min(cw, ch) > 0 else 0
            if area > 500 and ratio > 2.0:
                c[:, :, 1] += int(h * 0.35)
                cv2.drawContours(marked, [c], -1, (0, 0, 255), 2)
                total += 1
        cv2.imwrite(f"crack_lines/{f}", marked)
    print(f"Stage 8 Done: Normal cracks detected = {total}")


# RUN FULL PIPELINE

print("Starting full pipeline...")
extract_frames(video_path)
remove_noise()
remove_shadow()
enhance_contrast()
sharpen_frames()
detect_potholes()
detect_edge_cracks()
detect_normal_cracks()
print("\nPipeline complete! All outputs saved.")

Starting full pipeline...
Stage 1 Done: 43 frames extracted
Stage 2 Done: Noise removal complete
Stage 3 Done: Shadow removal complete
Stage 4 Done: Contrast enhancement complete
Stage 5 Done: Motion blur sharpening complete
Stage 6 Done: Potholes detected = 10
Stage 7 Done: Edge cracks detected = 71
Stage 8 Done: Normal cracks detected = 17

Pipeline complete! All outputs saved.
